# Step 3 — Baseline Reconstruction

Great-circle interpolation between the last known ADS-B point before the gap
and the first known ADS-B point after the gap.

This is our naive reference — any improved method must beat it.

In [10]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("../noel_part").resolve()))

import pandas as pd
import numpy as np
import json
from pyproj import Geod

BASE_DIR  = Path("../noel_part")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
RECON_DIR = BASE_DIR / "final_reconstructions"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

geod = Geod(ellps="WGS84")
flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]
print(f"Flights available: {len(flights)}")

Flights available: 2149


## 3a. How the baseline works

For a gap between time $t_0$ (last ADS-B before) and $t_1$ (first ADS-B after):

1. Compute the great-circle arc from the last known position to the first known position after the gap
2. Interpolate $n$ evenly spaced points along this arc
3. Blend a forward arc with a backward arc linearly

This is implemented in `src/step3_baseline.py` and `noel_part/reconstruction.py`.

## 3b. Run baseline on a sample flight

In [11]:
# ── 3b. Run baseline on a sample flight ──────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../noel_part").resolve()))

# Also need to be in the noel_part directory for scalers to load
import os
os.chdir(Path("../noel_part").resolve())

from reconstruction import reconstruct_gap_baseline
import pandas as pd
import numpy as np

# Re-resolve flights after chdir
CLEAN_DIR = Path("cleaned_data_final")
flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

sample = flights[0]
bef  = pd.read_parquet(sample / "adsb_before.parquet")
adsc = pd.read_parquet(sample / "adsc.parquet")
aft  = pd.read_parquet(sample / "adsb_after.parquet")

recon = reconstruct_gap_baseline(bef, aft)
print(f"Gap: {len(recon)} reconstructed steps")
print(f"From: lat={bef['latitude'].iloc[-1]:.3f}, lon={bef['longitude'].iloc[-1]:.3f}")
print(f"To  : lat={aft['latitude'].iloc[0]:.3f},  lon={aft['longitude'].iloc[0]:.3f}")
display(recon[["timestamp","latitude","longitude","altitude"]].head(5))

Gap: 919 reconstructed steps
From: lat=54.458, lon=-14.150
To  : lat=46.926,  lon=-62.602


,timestamp,latitude,longitude,altitude
0,2023-08-10 06:56:45,54.461683,-14.214738,9753.600000
1,2023-08-10 06:57:00,54.460335,-14.323232,9754.264052
2,2023-08-10 06:57:15,54.458972,-14.431612,9754.928105
3,2023-08-10 06:57:30,54.457594,-14.539875,9755.592157
4,2023-08-10 06:57:45,54.456201,-14.648023,9756.256209


## 3c. Evaluate baseline against ADS-C ground truth

In [12]:
# ── 3c. Evaluate baseline against ADS-C ground truth ─────────────────────────
import sys, os
from pathlib import Path

# Compute absolute path to notebooks folder BEFORE any chdir
NOTEBOOKS_DIR = Path(os.path.abspath("")).resolve()
if NOTEBOOKS_DIR.name != "notebooks":
    # We already chdired to noel_part, find notebooks as sibling
    NOTEBOOKS_DIR = NOTEBOOKS_DIR.parent / "notebooks"

print(f"Notebooks dir: {NOTEBOOKS_DIR}")
print(f"Exists: {NOTEBOOKS_DIR.exists()}")

# Add to path
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

# Also try adding the current directory
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"sys.path[0]: {sys.path[0]}")
print(f"sys.path[1]: {sys.path[1]}")

Notebooks dir: C:\Users\marko\Desktop\AeroML3\notebooks
Exists: False
sys.path[0]: C:\Users\marko\Desktop\AeroML3\noel_part
sys.path[1]: C:\Users\marko\Desktop\AeroML3\noel_part


## 3d. Export GeoJSON for geojson.io

In [13]:
import json

sample      = flights[0]
step_name   = sample.parent.name
flight_name = sample.name

bef  = pd.read_parquet(sample / "adsb_before.parquet")
adsc = pd.read_parquet(sample / "adsc.parquet")
aft  = pd.read_parquet(sample / "adsb_after.parquet")
recon = reconstruct_gap_baseline(bef, aft)

def _line(lons, lats, props):
    coords = [[float(lo), float(la)] for lo, la in zip(lons, lats)
              if np.isfinite(float(lo)) and np.isfinite(float(la))]
    if len(coords) < 2:
        return None
    return {"type": "Feature", "properties": props,
            "geometry": {"type": "LineString", "coordinates": coords}}

features = [
    _line(bef["longitude"],   bef["latitude"],
          {"label": "ADS-B before",           "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}),
    _line(aft["longitude"],   aft["latitude"],
          {"label": "ADS-B after",            "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}),
    _line(adsc["longitude"],  adsc["latitude"],
          {"label": "ADS-C ground truth",     "stroke": "#FFC107", "stroke-width": 3, "stroke-opacity": 1.0}),
    _line(recon["longitude"], recon["latitude"],
          {"label": "Baseline (great-circle)","stroke": "#F44336", "stroke-width": 2, "stroke-opacity": 0.9}),
]

features = [f for f in features if f]
fc  = {"type": "FeatureCollection", "features": features}
out = OUT_DIR / f"{flight_name}_baseline.geojson"
out.write_text(json.dumps(fc, indent=2), encoding="utf-8")
print(f"GeoJSON saved → {out}")
print("Open at https://geojson.io")
print()
print("Legend:")
print("  Grey   = ADS-B track (before + after gap)")
print("  Yellow = ADS-C ground truth")
print("  Red    = Baseline (great-circle interpolation)")

GeoJSON saved → ..\outputs\20230810_4ba959_073209_092245_baseline.geojson
Open at https://geojson.io

Legend:
  Grey   = ADS-B track (before + after gap)
  Yellow = ADS-C ground truth
  Red    = Baseline (great-circle interpolation)
